# Google ADK: Documentation RAG Agent

This notebook recreates the documentation RAG agent using **Google ADK** (Agent Development Kit).

ADK integrates with Gemini models, uses plain Python methods as tools, and manages conversation history through sessions.

## Install dependency

```bash
uv add google-adk
```

### Short production reminder

For production, replace print-based callbacks with structured logs or tracing events. Include request/run IDs, tool name, latency, status, and redacted argument summaries. Do not log sensitive content or full tool arguments by default. Keep callbacks observational; enforce authorization and validation inside the tool execution boundary.


In [ ]:
from search_tools import create_search_tools
from common import AGENT_INSTRUCTIONS

search_tools = create_search_tools()

## Define tools

ADK can use ordinary Python methods directly. It inspects their signatures and docstrings automatically, so no decorator is required.

In [ ]:
tools = [
    search_tools.search,
    search_tools.get_file,
]

## Create the agent

ADK uses `instruction` (singular), unlike many other frameworks that use `instructions`.

The `description` field is useful when this agent participates in a multi-agent system.

In [ ]:
from google.adk.agents import Agent

doc_agent = Agent(
    name="doc_rag_agent",
    model="gemini-3-flash-preview",
    instruction=AGENT_INSTRUCTIONS,
    description="A documentation assistant that answers questions about Evidently",
    tools=tools,
)

## Create session and runner

ADK requires a session service and a session. The `Runner` connects the agent to the session service and executes the agent loop.

`InMemorySessionService` stores history in memory.

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

APP_NAME = "doc_rag_agent"
USER_ID = "user_001"

session_service = InMemorySessionService()

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
)

runner = Runner(
    agent=doc_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

## Run the agent

ADK does not provide a simple “run and return answer” method. Instead, `runner.run()` yields events one at a time.

These events can include user input, function calls, function responses, intermediate reasoning, and the final response.

In [ ]:
from google.genai import types

query = "how do I use evidently to monitor my machine learning models?"

content = types.Content(
    role="user",
    parts=[types.Part(text=query)],
)

events = []

for event in runner.run(
    user_id=USER_ID,
    session_id=session.id,
    new_message=content,
):
    events.append(event)

final_response = events[-1].content.parts[0].text
print(final_response)

## Inspect session events

ADK stores the conversation in the session. Each event has an `author` and content parts that may contain text, function calls, or function responses.

In [ ]:
session = await session_service.get_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=session.id,
)

for event in session.events:
    print(f"Author: {event.author}")

    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(f"  Text: {part.text[:300]}")

            if part.function_call:
                print(
                    f"  Function call: "
                    f"{part.function_call.name}({dict(part.function_call.args)})"
                )

            if part.function_response:
                print(
                    f"  Function response: "
                    f"{str(part.function_response.response)[:200]}..."
                )

    print()

## Follow-up conversation

Send the follow-up to the same session. ADK automatically includes and updates the conversation history.

In [ ]:
follow_up = "show me the code"

follow_up_content = types.Content(
    role="user",
    parts=[types.Part(text=follow_up)],
)

events2 = []

for event in runner.run(
    user_id=USER_ID,
    session_id=session.id,
    new_message=follow_up_content,
):
    events2.append(event)

final_response2 = events2[-1].content.parts[0].text
print(final_response2)

In [ ]:
session = await session_service.get_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=session.id,
)

for event in session.events[-6:]:
    print(f"Author: {event.author}")

    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(f"  Text: {part.text[:300]}")

            if part.function_call:
                print(
                    f"  Function call: "
                    f"{part.function_call.name}({dict(part.function_call.args)})"
                )

            if part.function_response:
                print(
                    f"  Function response: "
                    f"{str(part.function_response.response)[:200]}..."
                )

    print()

## Production-style observability

ADK’s event stream is naturally suited to structured logging. Log each function call as it arrives, including request/session IDs, tool name, redacted arguments, and status.

The callback below observes events only. It does not execute tools or enforce permissions.

In [ ]:
import time
import logging
import uuid

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
)

logger = logging.getLogger("google_adk_docs")


def summarize_args(args, limit: int = 120) -> str:
    text = str(args)
    return text if len(text) <= limit else text[:limit] + "..."


def log_adk_events(events, request_id: str) -> None:
    """Observational post-run logging for ADK events."""
    for event in events:
        if not event.content or not event.content.parts:
            continue

        for part in event.content.parts:
            if part.function_call:
                logger.info(
                    "agent_tool_call",
                    extra={
                        "event": "tool_call",
                        "request_id": request_id,
                        "session_id": session.id,
                        "author": event.author,
                        "tool_name": part.function_call.name,
                        "args_summary": summarize_args(
                            dict(part.function_call.args)
                        ),
                        "status": "requested",
                    },
                )


request_id = str(uuid.uuid4())
started = time.perf_counter()

observability_events = []

for event in runner.run(
    user_id=USER_ID,
    session_id=session.id,
    new_message=content,
):
    observability_events.append(event)

logger.info(
    "agent_run_completed",
    extra={
        "event": "agent_run_completed",
        "request_id": request_id,
        "session_id": session.id,
        "latency_ms": round((time.perf_counter() - started) * 1000, 2),
        "status": "success",
    },
)

log_adk_events(observability_events, request_id)

print(observability_events[-1].content.parts[0].text)

## Client production checklist

- Use authenticated user and tenant IDs in the session or request context.
- Keep tools read-only until authorization and audit trails are established.
- Validate arguments and check permissions inside the tool function.
- Log request/session IDs, tool name, latency, status, and redacted arguments.
- Add timeouts, retries, rate limits, and cost controls.
- Avoid logging full tool arguments or sensitive user content.
- Test follow-up conversations, unauthorized access, invalid inputs, and tool failures.